In [1]:
!pip install stable-baselines3[extra] gymnasium[atari] ale-py autorom[accept-rom-license] -q
!AutoROM --accept-license

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 11.4 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 13.5 MB/s eta 0:00:00
AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.


In [2]:
import ale_py
import gymnasium as gym
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)

env = make_atari_env("ALE/SpaceInvaders-v5", n_envs=1, seed=42)
env = VecFrameStack(env, n_stack=4)

obs = env.reset()
print("Environment loaded successfully!")
print("Observation shape:", obs.shape)
env.close()

2026-03-21 09:14:22.922556: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774084463.107059      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774084463.161512      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774084463.591048      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774084463.591095      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774084463.591098      55 computation_placer.cc:177] computation placer alr

Environment loaded successfully!
Observation shape: (1, 84, 84, 4)


In [3]:
import os
import csv
import numpy as np
import ale_py
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.env_util import make_atari_env

gym.register_envs(ale_py)

experiments = [
    {"name": "exp01_baseline",
     "policy": "CnnPolicy", "lr": 1e-4,  "gamma": 0.99,
     "batch_size": 32,  "eps_start": 1.0, "eps_end": 0.01,  "eps_fraction": 0.1},

    {"name": "exp02_high_lr",
     "policy": "CnnPolicy", "lr": 1e-3,  "gamma": 0.90,
     "batch_size": 128, "eps_start": 1.0, "eps_end": 0.05,  "eps_fraction": 0.2},

    {"name": "exp03_tiny_lr",
     "policy": "CnnPolicy", "lr": 1e-5,  "gamma": 0.999,
     "batch_size": 16,  "eps_start": 0.5, "eps_end": 0.001, "eps_fraction": 0.05},

    {"name": "exp04_low_gamma",
     "policy": "CnnPolicy", "lr": 5e-4,  "gamma": 0.95,
     "batch_size": 64,  "eps_start": 1.0, "eps_end": 0.1,   "eps_fraction": 0.3},

    {"name": "exp05_large_batch",
     "policy": "CnnPolicy", "lr": 1e-4,  "gamma": 0.999,
     "batch_size": 256, "eps_start": 1.0, "eps_end": 0.01,  "eps_fraction": 0.5},

    {"name": "exp06_low_lr",
     "policy": "CnnPolicy", "lr": 5e-5,  "gamma": 0.98,
     "batch_size": 32,  "eps_start": 0.8, "eps_end": 0.05,  "eps_fraction": 0.4},

    {"name": "exp07_aggressive_lr",
     "policy": "CnnPolicy", "lr": 1e-3,  "gamma": 0.999,
     "batch_size": 64,  "eps_start": 1.0, "eps_end": 0.001, "eps_fraction": 0.15},

    {"name": "exp08_balanced",
     "policy": "CnnPolicy", "lr": 2e-4,  "gamma": 0.97,
     "batch_size": 128, "eps_start": 1.0, "eps_end": 0.02,  "eps_fraction": 0.25},

    {"name": "exp09_mlp_policy",
     "policy": "MlpPolicy", "lr": 1e-4,  "gamma": 0.99,
     "batch_size": 32,  "eps_start": 1.0, "eps_end": 0.05,  "eps_fraction": 0.1},

    {"name": "exp10_best",
     "policy": "CnnPolicy", "lr": 3e-4,  "gamma": 0.99,
     "batch_size": 64,  "eps_start": 1.0, "eps_end": 0.01,  "eps_fraction": 0.3},
]

csv_results = []

for i, config in enumerate(experiments):
    print(f"\n{'='*50}")
    print(f"  EXPERIMENT {i+1}/10 — {config['name']}")
    print(f"  policy={config['policy']}, lr={config['lr']}, gamma={config['gamma']}")
    print(f"  batch={config['batch_size']}, eps_start={config['eps_start']}")
    print(f"  eps_end={config['eps_end']}, eps_fraction={config['eps_fraction']}")
    print(f"{'='*50}")

    save_path = f"/kaggle/working/{config['name']}.zip"
    log_dir = f"/kaggle/working/logs/{config['name']}"
    os.makedirs(log_dir, exist_ok=True)

    if os.path.exists(save_path):
        print(f"Already done, skipping...")
        continue

    env = make_atari_env(
        "ALE/SpaceInvaders-v5",
        n_envs=1,
        seed=42,
        monitor_dir=log_dir
    )
    env = VecFrameStack(env, n_stack=4)

    model = DQN(
        policy=config["policy"],
        env=env,
        learning_rate=config["lr"],
        gamma=config["gamma"],
        batch_size=config["batch_size"],
        exploration_initial_eps=config["eps_start"],
        exploration_final_eps=config["eps_end"],
        exploration_fraction=config["eps_fraction"],
        verbose=1
    )

    model.learn(total_timesteps=100_000)

    total_rewards = []
    for episode in range(3):
        obs = env.reset()
        done = False
        ep_reward = 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = env.step(action)
            ep_reward += reward
        total_rewards.append(float(ep_reward))

    avg_reward = round(float(np.mean(total_rewards)), 2)
    std_reward = round(float(np.std(total_rewards)), 2)

    model.save(save_path.replace(".zip", ""))

    csv_results.append({
        "member_name": "Theodora",
        "experiment_name": config["name"],
        "policy": config["policy"],
        "env_id": "ALE/SpaceInvaders-v5",
        "timesteps": 100000,
        "lr": config["lr"],
        "gamma": config["gamma"],
        "batch_size": config["batch_size"],
        "epsilon_start": config["eps_start"],
        "epsilon_end": config["eps_end"],
        "epsilon_decay_fraction": config["eps_fraction"],
        "eval_mean_reward": avg_reward,
        "eval_std": std_reward
    })

    print(f"\nExperiment {i+1} done! Avg Reward: {avg_reward} +/- {std_reward}")
    env.close()

csv_path = "/kaggle/working/hyperparameter_results.csv"
if csv_results:
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=csv_results[0].keys())
        writer.writeheader()
        writer.writerows(csv_results)
    print(f"\nResults saved to {csv_path}")

print("\nALL 10 EXPERIMENTS COMPLETE!")


  EXPERIMENT 1/10 — exp01_baseline
  policy=CnnPolicy, lr=0.0001, gamma=0.99
  batch=32, eps_start=1.0
  eps_end=0.01, eps_fraction=0.1
Using cuda device
Wrapping the env in a VecTransposeImage.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 56.46GB > 31.60GB
  warnings.warn(


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 744      |
|    ep_rew_mean      | 220      |
|    exploration_rate | 0.98     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 88       |
|    time_elapsed     | 2        |
|    total_timesteps  | 206      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0199   |
|    n_updates        | 26       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 692      |
|    ep_rew_mean      | 250      |
|    exploration_rate | 0.956    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 123      |
|    time_elapsed     | 3        |
|    total_timesteps  | 445      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0502   |
|    n_updates      

/tmp/ipykernel_55/3902899748.py:103: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  total_rewards.append(float(ep_reward))



Experiment 1 done! Avg Reward: 2.67 +/- 2.49

  EXPERIMENT 2/10 — exp02_high_lr
  policy=CnnPolicy, lr=0.001, gamma=0.9
  batch=128, eps_start=1.0
  eps_end=0.05, eps_fraction=0.2
Using cuda device
Wrapping the env in a VecTransposeImage.


...

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 851      |
|    ep_rew_mean      | 295      |
|    exploration_rate | 0.993    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 186      |
|    time_elapsed     | 1        |
|    total_timesteps  | 237      |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.0196   |
|    n_updates        | 34       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 609      |
|    ep_rew_mean      | 168      |
|    exploration_rate | 0.99     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 185      |
|    time_elapsed     | 1        |
|    total_timesteps  | 344      |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.0234   |
|    n_updates      

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 56.46GB > 25.27GB
  warnings.warn(


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 744      |
|    ep_rew_mean      | 220      |
|    exploration_rate | 0.996    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 165      |
|    time_elapsed     | 1        |
|    total_timesteps  | 216      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0526   |
|    n_updates        | 28       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 692      |
|    ep_rew_mean      | 188      |
|    exploration_rate | 0.991    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 157      |
|    time_elapsed     | 2        |
|    total_timesteps  | 435      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0274   |
|    n_updates      

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 56.46GB > 25.31GB
  warnings.warn(


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.05e+03 |
|    ep_rew_mean      | 280      |
|    exploration_rate | 0.972    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 212      |
|    time_elapsed     | 1        |
|    total_timesteps  | 297      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0179   |
|    n_updates        | 49       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 788      |
|    ep_rew_mean      | 168      |
|    exploration_rate | 0.955    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 212      |
|    time_elapsed     | 2        |
|    total_timesteps  | 470      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0258   |
|    n_updates      

In [4]:
import csv

csv_path = "/kaggle/working/hyperparameter_results.csv"

print("\n" + "="*100)
print("        HYPERPARAMETER TUNING RESULTS - SPACE INVADERS")
print("="*100)
print(f"{'Exp':<5} {'Name':<25} {'Policy':<12} {'LR':<8} {'Gamma':<7} {'Batch':<7} {'eps_end':<9} {'eps_frac':<10} {'Avg Reward':<12} {'Std'}")
print("-"*100)

with open(csv_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"  {row['experiment_name']:<25} {row['policy']:<12} {row['lr']:<8} {row['gamma']:<7} {row['batch_size']:<7} {row['epsilon_end']:<9} {row['epsilon_decay_fraction']:<10} {row['eval_mean_reward']:<12} {row['eval_std']}")

print("="*100)


        HYPERPARAMETER TUNING RESULTS - SPACE INVADERS
Exp   Name                      Policy       LR       Gamma   Batch   eps_end   eps_frac   Avg Reward   Std
----------------------------------------------------------------------------------------------------
  exp01_baseline            CnnPolicy    0.0001   0.99    32      0.01      0.1        2.67         2.49
  exp02_high_lr             CnnPolicy    0.001    0.9     128     0.05      0.2        5.0          6.38
  exp03_tiny_lr             CnnPolicy    1e-05    0.999   16      0.001     0.05       5.67         0.94
  exp04_low_gamma           CnnPolicy    0.0005   0.95    64      0.1       0.3        2.33         1.7
  exp05_large_batch         CnnPolicy    0.0001   0.999   256     0.01      0.5        11.33        8.99
  exp06_low_lr              CnnPolicy    5e-05    0.98    32      0.05      0.4        2.33         2.62
  exp07_aggressive_lr       CnnPolicy    0.001    0.999   64      0.001     0.15       2.33         1.25
 

In [4]:
import ale_py
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.callbacks import EvalCallback

gym.register_envs(ale_py)

env = make_atari_env(
    "ALE/SpaceInvaders-v5",
    n_envs=1,
    seed=42,
    monitor_dir="/kaggle/working/logs/final_model"
)
env = VecFrameStack(env, n_stack=4)

eval_env = make_atari_env("ALE/SpaceInvaders-v5", n_envs=1, seed=42)
eval_env = VecFrameStack(eval_env, n_stack=4)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="/kaggle/working/best_model/",
    log_path="/kaggle/working/logs/final_model/",
    eval_freq=10000,
    verbose=1
)

final_model = DQN(
    policy="CnnPolicy",
    env=env,
    learning_rate=1e-4,
    gamma=0.999,
    batch_size=64,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,
    exploration_fraction=0.5,
    verbose=1,
    tensorboard_log="/kaggle/working/tensorboard_logs/"
)

final_model.learn(
    total_timesteps=500_000,
    callback=eval_callback
)

final_model.save("/kaggle/working/dqn_model")
print("Final model saved!")
env.close()
eval_env.close()



Using cuda device
Wrapping the env in a VecTransposeImage.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 56.46GB > 31.85GB
  warnings.warn(


Logging to /kaggle/working/tensorboard_logs/DQN_2


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7bb48697c4d0> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7bb48697c050>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 744      |
|    ep_rew_mean      | 220      |
|    exploration_rate | 0.999    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 92       |
|    time_elapsed     | 2        |
|    total_timesteps  | 216      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0549   |
|    n_updates        | 28       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 692      |
|    ep_rew_mean      | 188      |
|    exploration_rate | 0.998    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 117      |
|    time_elapsed     | 3        |
|    total_timesteps  | 386      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0614   |
|    n_updates      

In [5]:
import os

print("="*50)
print("FILES IN /kaggle/working/")
print("="*50)
for f in sorted(os.listdir("/kaggle/working/")):
    path = f"/kaggle/working/{f}"
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f"  {f:<40} {size:.1f} MB")
    else:
        print(f"  {f}/ (folder)")
print("="*50)

FILES IN /kaggle/working/
  .virtual_documents/ (folder)
  best_model/ (folder)
  dqn_model.zip                            26.0 MB
  exp01_baseline.zip                       26.0 MB
  exp02_high_lr.zip                        26.0 MB
  exp03_tiny_lr.zip                        26.0 MB
  exp04_low_gamma.zip                      26.0 MB
  exp05_large_batch.zip                    26.0 MB
  exp06_low_lr.zip                         26.0 MB
  exp07_aggressive_lr.zip                  26.0 MB
  exp08_balanced.zip                       26.0 MB
  exp09_mlp_policy.zip                     27.9 MB
  exp10_best.zip                           26.0 MB
  hyperparameter_results.csv               0.0 MB
  logs/ (folder)
  tensorboard_logs/ (folder)
